# Phase 15 (CLINICALLY-GROUNDED FINAL) — ED/ES Keyframe Model

**What changed from notebook 13:** the 8 evenly-spaced frames are replaced with exactly the two clinically-defined keyframes -- End-Diastole (ED, maximum LV volume) and End-Systole (ES, minimum LV volume) -- extracted directly from expert tracings in notebook 14. This matches how EF is actually computed clinically (Simpson's method from ED and ES volumes) and is directly defensible to a reviewer, rather than relying on a heuristic sampling assumption.

**Architecture change:** the shared-weight backbone processes the ED and ES frames separately, producing two feature vectors `f_ED` and `f_ES`. Instead of only mean-pooling (which would blur the two informative extremes together), the model uses the concatenation `[f_ED, f_ES, f_ED - f_ES]` -- the third term is an explicit, learnable contraction/motion signal, directly analogous to the volume difference clinicians use to compute EF.

**This is the true final, clinically-defensible model.** Requires notebook 14's ED/ES extraction to be complete first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, mean_absolute_error

PROJECT_ROOT = '/content/drive/MyDrive/echonet_project'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic', 'EchoNet-Dynamic')
FULL_CSV = os.path.join(DATA_DIR, 'FileList.csv')
EDES_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data/edes', 'frames')
quality_csv_path = os.path.join(PROJECT_ROOT, 'data/full', 'quality_scores.csv')
PRETRAINED_BACKBONE_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', 'simclr_pretrain_v1', 'backbone_only.pt')

assert os.path.isdir(EDES_CACHE_DIR) and len(os.listdir(EDES_CACHE_DIR)) > 9000, \
    'Run 14_ed_es_extraction.ipynb to completion first.'

df = pd.read_csv(FULL_CSV)
quality_df = pd.read_csv(quality_csv_path)
ef_col = 'EF' if 'EF' in df.columns else [c for c in df.columns if 'ef' in c.lower()][0]
filename_col = 'FileName' if 'FileName' in df.columns else [c for c in df.columns if 'file' in c.lower() and 'name' in c.lower()][0]
HFrEF_THRESHOLD = 40

for col in ['sharpness', 'contrast']:
    lo, hi = quality_df[col].quantile(0.02), quality_df[col].quantile(0.98)
    quality_df[col + '_norm'] = ((quality_df[col] - lo) / max(hi - lo, 1e-6)).clip(0, 1)
quality_df['quality_score'] = quality_df[['sharpness_norm', 'contrast_norm']].mean(axis=1)
df = df.merge(quality_df[[filename_col, 'quality_score']], on=filename_col, how='left')

valid_mask = df[filename_col].apply(lambda f: os.path.exists(
    os.path.join(EDES_CACHE_DIR, (str(f) if str(f).endswith('.avi') else str(f)+'.avi').replace('.avi', '.npy'))))
clean_df = df[valid_mask & df['quality_score'].notna()].reset_index(drop=True)
train_df = clean_df[clean_df['Split'].str.upper() == 'TRAIN'].reset_index(drop=True)
val_df = clean_df[clean_df['Split'].str.upper() == 'VAL'].reset_index(drop=True)
test_df = clean_df[clean_df['Split'].str.upper() == 'TEST'].reset_index(drop=True)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)} (test untouched until the final cell)')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: enable GPU before running this notebook.')

In [ ]:
# Reproducibility: fix random seeds (used for all runs from this version onward)
import random
import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Note: GPU kernels (cuDNN) can still be non-deterministic, so re-runs may differ slightly in the last digits.

## ED/ES dataset

In [ ]:
class EchoEDESDataset(Dataset):
    def __init__(self, df, frame_dir, filename_col, ef_col, train=True):
        self.df = df.reset_index(drop=True)
        self.frame_dir = frame_dir
        self.filename_col = filename_col
        self.ef_col = ef_col
        self.train = train
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = str(row[self.filename_col])
        if not fname.endswith('.avi'):
            fname = fname + '.avi'
        pair = np.load(os.path.join(self.frame_dir, fname.replace('.avi', '.npy')))
        pair = torch.from_numpy(pair).permute(0, 3, 1, 2).float() / 255.0
        pair = F.interpolate(pair, size=(112, 112), mode='bilinear', align_corners=False)
        if self.train and torch.rand(1).item() < 0.5:
            pair = torch.flip(pair, dims=[3])
        pair = (pair - self.mean) / self.std
        ef = float(row[self.ef_col])
        label = 1.0 if ef < HFrEF_THRESHOLD else 0.0
        quality = float(row['quality_score']) if not pd.isna(row['quality_score']) else 0.5
        return pair, torch.tensor(ef, dtype=torch.float32), torch.tensor(label, dtype=torch.float32), torch.tensor(quality, dtype=torch.float32)

BATCH_SIZE = 24
train_ds = EchoEDESDataset(train_df, EDES_CACHE_DIR, filename_col, ef_col, train=True)
val_ds = EchoEDESDataset(val_df, EDES_CACHE_DIR, filename_col, ef_col, train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## Model: shared-weight ED/ES backbone + explicit contraction-delta feature + FiLM + CQR heads

In [ ]:
class QualityFiLM(nn.Module):
    def __init__(self, feat_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, 2 * feat_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
        self.feat_dim = feat_dim

    def forward(self, quality_score):
        params = self.net(quality_score.unsqueeze(-1))
        gamma, beta = params[:, :self.feat_dim], params[:, self.feat_dim:]
        return 1.0 + gamma, beta

class EchoEDESCQR(nn.Module):
    def __init__(self, dropout_p=0.2, pretrained_backbone_path=None):
        super().__init__()
        backbone = models.resnet18(weights=None)
        backbone.fc = nn.Identity()
        if pretrained_backbone_path:
            backbone.load_state_dict(torch.load(pretrained_backbone_path, map_location='cpu'))
        self.backbone = backbone
        combined_dim = 512 * 3
        self.film = QualityFiLM(feat_dim=combined_dim)
        self.dropout = nn.Dropout(p=dropout_p)
        self.ef_mean_head = nn.Linear(combined_dim, 1)
        self.ef_q_lo_head = nn.Linear(combined_dim, 1)
        self.ef_q_hi_head = nn.Linear(combined_dim, 1)
        self.cls_head = nn.Linear(combined_dim, 1)

    def forward(self, pair, quality_score):
        ed = pair[:, 0]
        es = pair[:, 1]
        f_ed = self.backbone(ed)
        f_es = self.backbone(es)
        feats = torch.cat([f_ed, f_es, f_ed - f_es], dim=1)
        gamma, beta = self.film(quality_score)
        feats = gamma * feats + beta
        feats = self.dropout(feats)
        ef_mean = self.ef_mean_head(feats).squeeze(-1)
        q_lo = self.ef_q_lo_head(feats).squeeze(-1)
        q_hi = self.ef_q_hi_head(feats).squeeze(-1)
        cls_logit = self.cls_head(feats).squeeze(-1)
        return ef_mean, q_lo, q_hi, cls_logit

def pinball_loss(y_true, y_pred, tau):
    diff = y_true - y_pred
    return torch.mean(torch.max(tau * diff, (tau - 1) * diff))

TAU_LO, TAU_HI = 0.05, 0.95

## Train (checkpointed, resumable)

In [ ]:
EXP_NAME = 'edes_cqr_v1'
model = EchoEDESCQR(pretrained_backbone_path=PRETRAINED_BACKBONE_PATH).to(device)
with torch.no_grad():
    mean_ef = float(train_df[ef_col].mean())
    std_ef = float(train_df[ef_col].std())
    model.ef_mean_head.bias.fill_(mean_ef); model.ef_mean_head.weight.zero_()
    model.ef_q_lo_head.bias.fill_(mean_ef - std_ef); model.ef_q_lo_head.weight.zero_()
    model.ef_q_hi_head.bias.fill_(mean_ef + std_ef); model.ef_q_hi_head.weight.zero_()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
pos_rate = (train_df[ef_col] < HFrEF_THRESHOLD).mean()
pos_weight = torch.tensor([(1 - pos_rate) / max(pos_rate, 1e-6)], device=device)
cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

STATE_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', EXP_NAME, 'latest.pt')
os.makedirs(os.path.dirname(STATE_PATH), exist_ok=True)
start_epoch, resume_step = 0, 0
if os.path.exists(STATE_PATH):
    state = torch.load(STATE_PATH, map_location=device)
    model.load_state_dict(state['model_state']); optimizer.load_state_dict(state['optimizer_state'])
    start_epoch, resume_step = state['epoch'], state['step_in_epoch']
    print(f'[{EXP_NAME}] resumed from epoch={start_epoch}')

N_EPOCHS = 10
CHECKPOINT_EVERY_MINUTES = 15
last_ckpt_time = time.time()

def evaluate(model, loader):
    model.eval()
    ef_true_all, ef_pred_all, cls_true_all, cls_prob_all = [], [], [], []
    with torch.no_grad():
        for pair, ef_true, cls_true, quality in loader:
            pair, quality = pair.to(device), quality.to(device)
            ef_mean, q_lo, q_hi, cls_logit = model(pair, quality)
            ef_true_all.extend(ef_true.numpy()); ef_pred_all.extend(ef_mean.cpu().numpy())
            cls_true_all.extend(cls_true.numpy()); cls_prob_all.extend(torch.sigmoid(cls_logit).cpu().numpy())
    mae = mean_absolute_error(ef_true_all, ef_pred_all)
    try:
        auc = roc_auc_score(cls_true_all, cls_prob_all)
    except ValueError:
        auc = float('nan')
    return {'mae': mae, 'auc': auc}

for epoch in range(start_epoch, N_EPOCHS):
    model.train()
    for step, (pair, ef_true, cls_true, quality) in enumerate(train_loader):
        if epoch == start_epoch and step < resume_step:
            continue
        pair, ef_true, cls_true, quality = pair.to(device), ef_true.to(device), cls_true.to(device), quality.to(device)
        ef_mean, q_lo, q_hi, cls_logit = model(pair, quality)
        loss = (F.l1_loss(ef_mean, ef_true)
                + pinball_loss(ef_true, q_lo, TAU_LO)
                + pinball_loss(ef_true, q_hi, TAU_HI)
                + cls_criterion(cls_logit, cls_true))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        if (time.time() - last_ckpt_time) > CHECKPOINT_EVERY_MINUTES * 60:
            torch.save({'epoch': epoch, 'step_in_epoch': step + 1, 'model_state': model.state_dict(),
                        'optimizer_state': optimizer.state_dict()}, STATE_PATH)
            last_ckpt_time = time.time()
    resume_step = 0
    torch.save({'epoch': epoch + 1, 'step_in_epoch': 0, 'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict()}, STATE_PATH)
    val_metrics = evaluate(model, val_loader)
    print(f'Epoch {epoch}: val_mae={val_metrics["mae"]:.3f} val_auc={val_metrics["auc"]:.3f}')

print('Training complete (or reached N_EPOCHS). Re-run to resume after a disconnect.')
print('Compare val_mae here to the 8-frame model\'s ~6.3-6.4 to see whether clinically-grounded keyframes help further.')

## CQR calibration on VAL

In [ ]:
def predict_all(model, loader):
    model.eval()
    ef_true_all, ef_mean_all, q_lo_all, q_hi_all, cls_true_all, cls_prob_all = [], [], [], [], [], []
    with torch.no_grad():
        for pair, ef_true, cls_true, quality in loader:
            pair, quality = pair.to(device), quality.to(device)
            ef_mean, q_lo, q_hi, cls_logit = model(pair, quality)
            ef_true_all.extend(ef_true.numpy()); ef_mean_all.extend(ef_mean.cpu().numpy())
            q_lo_all.extend(q_lo.cpu().numpy()); q_hi_all.extend(q_hi.cpu().numpy())
            cls_true_all.extend(cls_true.numpy()); cls_prob_all.extend(torch.sigmoid(cls_logit).cpu().numpy())
    return {
        'ef_true': np.array(ef_true_all), 'ef_mean': np.array(ef_mean_all),
        'q_lo': np.array(q_lo_all), 'q_hi': np.array(q_hi_all),
        'cls_true': np.array(cls_true_all), 'cls_prob': np.array(cls_prob_all),
    }

val_results = predict_all(model, val_loader)

rng = np.random.default_rng(42)
n_val = len(val_results['ef_true'])
perm = rng.permutation(n_val)
calib_idx, calibcheck_idx = perm[:n_val // 2], perm[n_val // 2:]

ALPHA = 0.1
E_calib = np.maximum(val_results['q_lo'][calib_idx] - val_results['ef_true'][calib_idx],
                      val_results['ef_true'][calib_idx] - val_results['q_hi'][calib_idx])
Q_hat = np.quantile(E_calib, min(1.0, (1 - ALPHA) * (1 + 1 / len(calib_idx))))
print(f'CQR correction Q_hat: {Q_hat:.3f}')

check_lo = val_results['q_lo'][calibcheck_idx] - Q_hat
check_hi = val_results['q_hi'][calibcheck_idx] + Q_hat
check_covered = (val_results['ef_true'][calibcheck_idx] >= check_lo) & (val_results['ef_true'][calibcheck_idx] <= check_hi)
print(f'Held-out calibration check coverage (target {(1-ALPHA)*100:.0f}%): {check_covered.mean()*100:.1f}%')
print(f'Mean interval width: {(check_hi - check_lo).mean():.2f} EF percentage points')

## THE CLINICALLY-GROUNDED, TRUE FINAL TEST EVALUATION

In [ ]:
test_ds = EchoEDESDataset(test_df, EDES_CACHE_DIR, filename_col, ef_col, train=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_results = predict_all(model, test_loader)

point_mae = mean_absolute_error(test_results['ef_true'], test_results['ef_mean'])
point_auc = roc_auc_score(test_results['cls_true'], test_results['cls_prob'])

test_lo = test_results['q_lo'] - Q_hat
test_hi = test_results['q_hi'] + Q_hat
test_covered = (test_results['ef_true'] >= test_lo) & (test_results['ef_true'] <= test_hi)
test_coverage = test_covered.mean()
test_mean_width = (test_hi - test_lo).mean()

print(f'TEST EF MAE: {point_mae:.3f}  (8-frame final was 6.356)')
print(f'TEST HFrEF AUC: {point_auc:.3f}  (8-frame final was 0.911)')
print(f'TEST CQR interval coverage (target 90%): {test_coverage*100:.1f}%')
print(f'TEST mean CQR interval width: {test_mean_width:.2f} EF percentage points')

N_BOOTSTRAP = 1000
rng2 = np.random.default_rng(123)
n_test = len(test_results['ef_true'])
boot_maes, boot_aucs = [], []
for _ in range(N_BOOTSTRAP):
    idx = rng2.integers(0, n_test, size=n_test)
    boot_maes.append(mean_absolute_error(test_results['ef_true'][idx], test_results['ef_mean'][idx]))
    try:
        boot_aucs.append(roc_auc_score(test_results['cls_true'][idx], test_results['cls_prob'][idx]))
    except ValueError:
        pass
mae_ci = (np.percentile(boot_maes, 2.5), np.percentile(boot_maes, 97.5))
auc_ci = (np.percentile(boot_aucs, 2.5), np.percentile(boot_aucs, 97.5))
print(f'\nTEST EF MAE: {point_mae:.3f} (95% CI: {mae_ci[0]:.3f}-{mae_ci[1]:.3f})')
print(f'TEST HFrEF AUC: {point_auc:.3f} (95% CI: {auc_ci[0]:.3f}-{auc_ci[1]:.3f})')

final_edes_results = {
    'test_mae': point_mae, 'test_mae_ci': mae_ci, 'test_auc': point_auc, 'test_auc_ci': auc_ci,
    'cqr_target_coverage': 1 - ALPHA, 'cqr_empirical_test_coverage': float(test_coverage),
    'cqr_mean_interval_width': float(test_mean_width),
}
with open(os.path.join(PROJECT_ROOT, 'logs', 'final_edes_test_results.json'), 'w') as f:
    json.dump(final_edes_results, f, indent=2)
print('\nSaved to logs/final_edes_test_results.json -- THIS IS THE CLINICALLY-GROUNDED DEFINITIVE FINAL RESULT.')

## What to report

Compare directly to the 8-frame evenly-spaced result (6.356 / 0.911). Two honest outcomes are both reportable:
- **If ED/ES keyframes improve or match the 8-frame result with a stronger clinical justification**, this becomes the headline architecture for the paper -- defensible on both empirical and clinical-methodology grounds.
- **If 8 evenly-spaced frames still perform better**, that is also a valid finding: it would suggest the extra temporal context between the extremes (not just the two endpoints) carries useful signal beyond what ED/ES alone captures -- worth discussing explicitly rather than silently picking whichever number is larger.